In [1]:
from pydantic import BaseModel, Field
from typing import Literal, Optional, Union
from enum import Enum

class LayerType(str, Enum):
    conv2d = "conv2d"
    linear = "linear"
    batchnorm2d = "batchnorm2d"
    layernorm = "layernorm"
    activation = "activation"
    pooling = "pooling"
    dropout = "dropout"
    flatten = "flatten"
    residual_block = "residual_block"   # composite
    attention = "attention"             # composite, for transformer-ish papers later
    embedding = "embedding"

class ActivationFn(str, Enum):
    relu = "relu"
    gelu = "gelu"
    tanh = "tanh"
    sigmoid = "sigmoid"
    softmax = "softmax"
    leaky_relu = "leaky_relu"

class LayerSpec(BaseModel):
    id: str                                  # unique within model, for residual refs
    type: LayerType
    params: dict = Field(default_factory=dict)   # e.g. {"in_channels":3,"out_channels":64,"kernel_size":3}
    source_confidence: float = 1.0           # extraction confidence, drives diagnostic priority
    source_span: Optional[str] = None        # verbatim text this was extracted from (for audit)

class TrainingSpec(BaseModel):
    optimizer: Optional[str] = None
    learning_rate: Optional[float] = None
    lr_schedule: Optional[str] = None
    batch_size: Optional[int] = None
    epochs: Optional[int] = None
    loss_fn: Optional[str] = None
    weight_decay: Optional[float] = None
    dataset: Optional[str] = None
    train_val_split: Optional[str] = None

class ReportedResult(BaseModel):
    metric_name: str          # "accuracy", "F1", "top-1"
    metric_value: float
    dataset_split: str         # "test", "val"

class ModelIR(BaseModel):
    schema_version: str = "v1"
    paper_title: Optional[str] = None
    layers: list[LayerSpec]
    training: TrainingSpec
    reported_results: list[ReportedResult] = Field(default_factory=list)
    unsupported_elements: list[str] = Field(default_factory=list)  # things extraction couldn't map

In [3]:
section_4_3_text = """
4.3 Models without sentiment scores

Figure 7 illustrates our initial classification framework. As detailed in Section 3, our
initial dataset consists of 52 numerical and 108 one-hot-encoded categorical variables.
We employ PCA algorithm to represent the variables with 20 dimensions. Similarly, the
dimension of the categorical variables is reduced to 45 dimensions by applying MCA
algorithm. In the next step, SMOTE algorithm is implemented on the reduced dataset to
balance the number of classes by oversampling the minority class. The reduced and
oversampled dataset is then passed to a Feedforward Neural Network (FFNN) in order to
predict the success/failure of the deals.

According to the results of the Bayesian Optimization hyperparameter tuning, one hidden
layer with 64 neurons is used for NN-Recall model, where we use selu (scaled exponential
linear unit) activation function and binary crossentropy loss function. NN-Accuracy model
is defined by two hidden layers with 128 and 8 neurons, respectively. Relu (rectified
linear unit) activation function is used in both layers, and binary cross-entropy function
is chosen as the loss function. Similarly, two hidden layers with 256 and 8 neurons with
relu activation function are used for NN-F1 model. In this model, F1 Loss is chosen as the
loss function.
"""

In [4]:
from extraction_agent import extract_ir

ir = extract_ir(section_4_3_text, paper_title="Predicting Status of Pre and Post M&A Deals")
print(ir.model_dump_json(indent=2))

{
  "schema_version": "v1",
  "paper_title": "Predicting Status of Pre and Post M&A Deals",
  "layers": [
    {
      "id": "layer1",
      "type": "linear",
      "params": {
        "out_features": 64
      },
      "source_confidence": 1.0,
      "source_span": "one hidden layer with 64 neurons"
    },
    {
      "id": "layer2",
      "type": "activation",
      "params": {
        "fn": "gelu"
      },
      "source_confidence": 1.0,
      "source_span": "selu (scaled exponential linear unit) activation function"
    }
  ],
  "training": {
    "optimizer": null,
    "learning_rate": null,
    "lr_schedule": null,
    "batch_size": null,
    "epochs": null,
    "loss_fn": "binary crossentropy",
    "weight_decay": null,
    "dataset": null,
    "train_val_split": null
  },
  "reported_results": [],
  "unsupported_elements": [
    "additional model variant 'NN-Accuracy' described in text but not extracted",
    "additional model variant 'NN-F1' described in text but not extracted",


In [7]:
ir.layers[0].params["in_features"] = 65  # from PCA(20) + MCA(45), stated elsewhere in the paper (Section 3)

model = build_model(ir)
print(model)

GeneratedModel(
  (net): Sequential(
    (0): Linear(in_features=65, out_features=64, bias=True)
    (1): GELU(approximate='none')
  )
)


In [9]:
from synthetic_data import generate_ma_deal_dataset
from ir_schema import ModelIR, LayerSpec, TrainingSpec, ReportedResult
from train import train_model
from diagnose_agent import diagnose

data = generate_ma_deal_dataset()

ir = ModelIR(
    layers=[
        LayerSpec(id='l1', type='linear', params={'out_features':128}),
        LayerSpec(id='a1', type='activation', params={'fn':'relu'}),
        LayerSpec(id='l2', type='linear', params={'in_features':128,'out_features':8}),
        LayerSpec(id='a2', type='activation', params={'fn':'relu'}),
        LayerSpec(id='l3', type='linear', params={'in_features':8,'out_features':1}),
        LayerSpec(id='out_act', type='activation', params={'fn':'sigmoid'}),
    ],
    training=TrainingSpec(loss_fn='binary_cross_entropy', epochs=5, batch_size=64),
    reported_results=[ReportedResult(metric_name='accuracy', metric_value=0.88, dataset_split='test')],
)

result = train_model(ir, data['X_train'], data['y_train'], data['X_val'], data['y_val'], verbose=True)

verdict = diagnose(ir, result, used_synthetic_data=True)
print(verdict.label)
print(verdict.summary)

Epoch 1/5 - loss: 0.3167 - acc: 0.8520 - val_loss: 0.1239 - val_acc: 0.9464
Epoch 2/5 - loss: 0.0984 - acc: 0.9578 - val_loss: 0.1251 - val_acc: 0.9366
Epoch 3/5 - loss: 0.0794 - acc: 0.9663 - val_loss: 0.1256 - val_acc: 0.9366
Epoch 4/5 - loss: 0.0685 - acc: 0.9719 - val_loss: 0.1238 - val_acc: 0.9475
Epoch 5/5 - loss: 0.0617 - acc: 0.9756 - val_loss: 0.1446 - val_acc: 0.9377
not_comparable
The reproducibility check resulted in a **not_comparable** verdict. The experiment was run on synthetic or substitute data rather than the paper’s original proprietary dataset. Consequently, any accuracy obtained only confirms that the training pipeline works, but it does not allow comparison with the paper’s reported results. No direct numerical comparison can be made.


In [10]:
from diagnose_agent import diagnose

verdict = diagnose(ir, result, used_synthetic_data=True)  # True since you're using synthetic_data.py
print(verdict.label)
print(verdict.summary)

not_comparable
The reproducibility check resulted in a **not_comparable** verdict. The experiment was run on synthetic or substitute data rather than the paper’s original proprietary dataset. Consequently, any accuracy obtained only confirms that the training pipeline works, but it does not allow comparison with the paper’s reported results. No direct metric values are available for comparison.


In [24]:
!python edgar_dataset_builder.py --target-deals 200 --start-date 2019-01-01 --end-date 2023-12-31 --out ma_deals.csv

Searching EDGAR full-text search for merger-agreement 8-Ks (2019-01-01 to 2023-12-31)...
  requesting EDGAR full-text search, from=0 ...
  got 100 hits on this page
  requesting EDGAR full-text search, from=100 ...
  EDGAR returned an error at from=100 (500 Server Error: Internal Server Error for url: https://efts.sec.gov/LATEST/search-index?q=%22merger+agreement%22&forms=8-K&dateRange=custom&startdt=2019-01-01&enddt=2023-12-31&from=100&sort=filedAt%3Adesc) -- stopping pagination here, keeping the 100 hits already collected. This is a known EDGAR deep-pagination limitation, not a bug in this script.
Found 100 candidate announcement filings. Resolving outcomes...
  processed 10/100 candidates, 10 labeled deals so far
  processed 20/100 candidates, 20 labeled deals so far
  processed 30/100 candidates, 30 labeled deals so far
  processed 40/100 candidates, 40 labeled deals so far
  processed 50/100 candidates, 50 labeled deals so far
  processed 60/100 candidates, 60 labeled deals so far

In [3]:
!python generate_dashboard_data.py

Step 1/5: Extracting IR from paper text...
Step 2/5: Building model from IR...
Step 3/5: Loading dataset...
Step 4/5: Training...
Epoch 1/20 - loss: 0.6919 - acc: 0.5125 - val_loss: 0.6872 - val_acc: 0.6250
Epoch 2/20 - loss: 0.6852 - acc: 0.6375 - val_loss: 0.6816 - val_acc: 0.5750
Epoch 3/20 - loss: 0.6794 - acc: 0.5938 - val_loss: 0.6758 - val_acc: 0.5750
Epoch 4/20 - loss: 0.6741 - acc: 0.5875 - val_loss: 0.6696 - val_acc: 0.5750
Epoch 5/20 - loss: 0.6682 - acc: 0.5875 - val_loss: 0.6631 - val_acc: 0.5750
Epoch 6/20 - loss: 0.6613 - acc: 0.5875 - val_loss: 0.6562 - val_acc: 0.5750
Epoch 7/20 - loss: 0.6547 - acc: 0.5875 - val_loss: 0.6485 - val_acc: 0.5750
Epoch 8/20 - loss: 0.6470 - acc: 0.6062 - val_loss: 0.6404 - val_acc: 0.6250
Epoch 9/20 - loss: 0.6390 - acc: 0.6250 - val_loss: 0.6314 - val_acc: 0.6250
Epoch 10/20 - loss: 0.6301 - acc: 0.6500 - val_loss: 0.6217 - val_acc: 0.6500
Epoch 11/20 - loss: 0.6207 - acc: 0.6625 - val_loss: 0.6109 - val_acc: 0.6750
Epoch 12/20 - loss: 0